# Selenium Web Scraping

### Core Concepts
- `driver` — sends Python commands to ChromeDriver
- `ChromeDriver` — translates driver commands into Chrome's language
- `ChromeDriverManager` — downloads the correct ChromeDriver version automatically
- `Service` — starts and stops the ChromeDriver process

### Flow
```
Your Python code (driver)  →  ChromeDriver (translates)  →  Chrome (acts)
```

# 1. Setup — Imports and get_driver()

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd

# Normal driver — opens visible Chrome window
def get_driver():
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()))

# Headless driver — Chrome runs invisibly, no window opens
def get_headless_driver():
    options = Options()
    options.add_argument('--headless')
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Test setup
driver = get_driver()
print('Browser opened successfully')
driver.quit()
print('Browser closed')

# 2. Navigation

- `driver.get(url)` — open a URL
- `driver.back()` — go back
- `driver.forward()` — go forward
- `driver.refresh()` — refresh page
- `driver.quit()` — close browser and ChromeDriver

In [ ]:
driver = get_driver()

driver.get('https://google.com')            # open Google
time.sleep(2)
driver.get('https://news.ycombinator.com')  # open Hacker News
time.sleep(2)
driver.back()                               # go back to Google
time.sleep(2)
driver.forward()                            # go forward to Hacker News
time.sleep(2)

driver.quit()

# 3. Finding Elements

- `find_element()` — finds ONE element, first match
- `find_elements()` — finds ALL matching elements, returns list

### By Options
- `By.CSS_SELECTOR` — most used, same syntax as BeautifulSoup select()
- `By.ID` — find by id attribute
- `By.CLASS_NAME` — find by class
- `By.TAG_NAME` — find by tag

### CSS Selector Rules
- `.classname a` — any a anywhere inside element (any depth)
- `.classname > a` — only direct child a (one level down)

In [ ]:
driver = get_driver()
driver.get('https://news.ycombinator.com')

# find ONE element
first_title = driver.find_element(By.CSS_SELECTOR, '.titleline > a')
print('First title:', first_title.text)

# find ALL elements
all_titles = driver.find_elements(By.CSS_SELECTOR, '.titleline > a')
print('Total titles found:', len(all_titles))

# print first 5
title_texts = [t.text for t in all_titles]  # extract text before quit()
driver.quit()

for title in title_texts[:5]:
    print(title)

# 4. Extracting Data — .text and .get_attribute()

- `.text` — gets visible text inside the tag (auto strips whitespace)
- `.get_attribute('href')` — gets attribute value, returns None if missing (safe)

**Important:** WebElements are live connections to Chrome. Extract all data BEFORE driver.quit()

In [ ]:
driver = get_driver()
driver.get('https://news.ycombinator.com')

titles = driver.find_elements(By.CSS_SELECTOR, '.titleline > a')

# extract everything before quit()
data = []
for title in titles[:5]:
    text = title.text
    link = title.get_attribute('href')
    data.append({'title': text, 'link': link})

driver.quit()  # safe to quit now — data is in plain Python list

for item in data:
    print(f"{item['title']} - {item['link']}")

# 5. Interacting with Pages — click(), send_keys(), clear()

- `.click()` — clicks a button, link, or checkbox
- `.send_keys('text')` — types text into input field
- `.send_keys(Keys.RETURN)` — presses Enter key
- `.clear()` — clears existing text from input field

In [ ]:
driver = get_driver()
driver.get('https://duckduckgo.com')

# find search box and type
search_box = driver.find_element(By.CSS_SELECTOR, "input[name='q']")
search_box.clear()                      # clear any existing text
search_box.send_keys('data analyst jobs')  # type search query
search_box.send_keys(Keys.RETURN)       # press Enter

time.sleep(3)  # wait for results to load
print('Current URL:', driver.current_url)

driver.quit()

# 6. Waits — Implicit and Explicit

### Why waits are needed
Selenium moves faster than pages load. Without waits it tries to find elements before they exist — crashes.

### Implicit Wait
- Set once, applies to every find_element() call globally
- Waits up to X seconds before crashing

### Explicit Wait
- Waits for a specific condition on a specific element
- Smarter and more reliable than implicit wait
- Common conditions: `presence_of_element_located`, `visibility_of_element_located`, `element_to_be_clickable`

In [ ]:
driver = get_driver()
driver.get('https://news.ycombinator.com')

# Implicit wait — set once, works everywhere
driver.implicitly_wait(10)

title = driver.find_element(By.CSS_SELECTOR, '.titleline > a')
print('Implicit wait result:', title.text)

driver.quit()

In [ ]:
driver = get_driver()
driver.get('https://news.ycombinator.com')

# Explicit wait — wait for specific element with specific condition
wait = WebDriverWait(driver, 10)  # max 10 seconds

# wait until element is present in HTML
title = wait.until(
    EC.presence_of_element_located((By.CSS_SELECTOR, '.titleline > a'))
)
print('Explicit wait result:', title.text)

driver.quit()

# 7. Headless Mode and Scrolling

### Headless Mode
Chrome runs invisibly in background — no window opens. Useful for running scrapers without disturbance.

### execute_script() — Scrolling
Some sites load more content as you scroll (infinite scroll). Use JavaScript to scroll:
- `window.scrollTo(0, document.body.scrollHeight)` — scroll to bottom
- `window.scrollBy(0, 500)` — scroll down by 500 pixels

Always use time.sleep() after scrolling to wait for new content to load.

In [ ]:
# Headless mode — no visible window
driver = get_headless_driver()
driver.get('https://news.ycombinator.com')

titles = driver.find_elements(By.CSS_SELECTOR, '.titleline > a')
title_texts = [t.text for t in titles[:5]]

driver.quit()

print('Scraped in headless mode:')
for title in title_texts:
    print(title)

In [ ]:
# Scrolling — scroll to bottom to load more content
driver = get_driver()
driver.get('https://news.ycombinator.com')

# scroll to bottom of page
driver.execute_script('window.scrollTo(0, document.body.scrollHeight)')
time.sleep(2)  # wait for content to load after scroll

# scroll down by specific pixels
driver.execute_script('window.scrollBy(0, 500)')
time.sleep(1)

print('Scrolled successfully')
driver.quit()